# 🚗 YOLOv11s Domain Adaptation — Singapore Smart City

This notebook represents **Level 1 (Perception)** of the hierarchical ML system. 
We are performing **Domain Adaptation** by fine-tuning a base YOLOv11s model on our proprietary, geographically-local `sg_traffic_dataset` rather than using generic open-source data.

### Senior MLOps Features Enabled:
- 🧠 **Cosine Annealing** Learning Rate Scheduler (`cos_lr=True`).
- 🧩 **Mosaic Augmentation Fade-out** (`close_mosaic=10`) for late-stage fine-tuning.
- 📊 **Weights & Biases (W&B)** integration for enterprise-grade experiment tracking.
- ⚡ **AdamW Optimizer** with decoupled weight decay for superior generalization.

**Platform Target:** Kaggle (T4 x2 GPUs) or Google Colab (L4/A100).

In [ ]:
# 1. Install Enterprise Dependencies
!pip install -q ultralytics wandb mlflow

import os
import torch
import shutil
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Locate Proprietary Singapore Dataset (Zero-Leakage Temporal Split)

# Auto-detect environment (Kaggle vs Colab)
if os.path.exists("/kaggle/input"):
    print("🌍 Kaggle Environment Detected")
    # Assuming user uploaded the zip to Kaggle Datasets
    dataset_root = Path("/kaggle/working/sg_traffic")
    if not dataset_root.exists():
        # Search input for zip or extracted folder
        input_dir = Path("/kaggle/input")
        zips = list(input_dir.rglob("*.zip"))
        if zips:
            print(f"Extracting dataset from {zips[0]}...")
            import zipfile
            with zipfile.ZipFile(zips[0], "r") as zip_ref:
                zip_ref.extractall("/kaggle/working/")
        else:
            # Assume it is already unzipped by Kaggle dataset manager
            yaml_files = list(input_dir.rglob("data.yaml"))
            if yaml_files:
                dataset_root = yaml_files[0].parent
else:
    print("☁️ Google Colab Environment Detected")
    from google.colab import drive
    drive.mount("/content/drive")
    dataset_root = Path("/content/dataset/sg_traffic")
    if not dataset_root.exists():
        zip_path = "/content/drive/MyDrive/sg_smart_city/data/sg_traffic_dataset.zip"
        !unzip -q {zip_path} -d /content/dataset/

data_yaml = dataset_root / "data.yaml"
print(f"\n✅ Target YAML located at: {data_yaml}")
print(data_yaml.read_text())

In [ ]:
# 3. Fix YAML Paths dynamically for target environment
import yaml
with open(data_yaml, "r") as f:
    data = yaml.safe_load(f)
    
data["path"] = str(dataset_root)
with open(data_yaml, "w") as f:
    yaml.dump(data, f)

print("✅ YAML paths dynamically linked to current runtime.")

In [ ]:
# 4. Initialize MLOps Observability (Weights & Biases)
import wandb
from ultralytics import settings

# Use W&B for loss curves and system metrics
settings.update({"wandb": True})

try:
    wandb.login(anonymous="allow") # Or use your specific W&B key
except:
    print("W&B login skipped, continuing without MLOps dashboard.")

In [ ]:
# 5. Execute Domain Adaptation (Level 1 Inference Training)
from ultralytics import YOLO

# Start from the pretrained small network
model = YOLO("yolo11s.pt")

print("\n🚀 Initiating Senior ML Distributed Training Loop...")
results = model.train(
    data=str(data_yaml),
    epochs=75,
    imgsz=640,
    batch=32,            # Maximize GPU memory throughput
    patience=12,         # Early stopping
    
    # Advanced Optimization
    optimizer="AdamW",   # Adam with proper Weight Decay
    lr0=0.001,           # Initial LR
    lrf=0.01,            # Final LR fraction
    cos_lr=True,         # Cosine Annealing scheduler (Senior technique)
    weight_decay=0.01,
    warmup_epochs=3.0,
    
    # Geometric & Photometric Augmentations
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0,          # Heavy composite training
    close_mosaic=10,     # Turn off mosaic in final 10 epochs for localized refinement
    
    # MLOps
    project="sg-smart-city",
    name="v1_domain_adaptation",
    cache=True,
    verbose=True,
    save_period=10
)

print(f"\n✅ Training complete! Best weights saved to: {results.save_dir}/weights/best.pt")

In [ ]:
# 6. Evaluate and Export Production Artifacts
best_model_path = f"{results.save_dir}/weights/best.pt"
best_model = YOLO(best_model_path)

# Export to ONNX execution graph for Vercel/Docker inference server
print("\n⚡ Compiling to ONNX Computational Graph for CPU Server Inference...")
best_model.export(format="onnx", dynamic=True, simplify=True)

print("\n✅ ML Model Pipeline Complete. Download best.pt and best.onnx from the output directory.")